<h1 align="center">Lab_S5: Transformers: Discriminative fine-tuning and Low Rank Adaptation (LoRA) for text classification
<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- (replace with one name)
- (replace with another name)

In [ ]:
!pip install transformers --upgrade
!pip install datasets accelerate --upgrade
!pip install peft --upgrade
#!pip install jupyter --upgrade
#!pip install ipywidgets --upgrade

## Many libraries

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import  AutoTokenizer, AutoModelForSequenceClassification,  Trainer, TrainingArguments,  EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import random
import os
import pandas as pd
import sys
import tempfile
import time
from functools import partial

In [ ]:
# IF YOU USE GOOGLE COLAB
COLAB = True

In [ ]:
#
if COLAB is True:
  from google.colab import drive
  drive.mount('/content/drive')
  dataset_path = "/content/drive/MyDrive/LNR/EXIST/"
else:
  dataset_path = ".."
dataset_path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/LNR/EXIST/'

In [ ]:
DATASET_FILE = "EXIST2026_training"


LABELS_21 = ['NO', 'YES']
LABELS_22 = ['JUDGEMENTAL', 'DIRECT']
LABELS_23 = ['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']

MAPPING_21 = {"YES": 1, "NO": 0}
MAPPING_22 = {"DIRECT": 1, "JUDGEMENTAL": 0}

DEV_PART = 0.2

## Read dataset

In [ ]:
## Read and split the dataset for each task. Use the same code developed in the previous lab session - Lab_S4



## Set the seed

In [ ]:
def set_seed(seed=1234):
    """
    Sets the seed to make everything deterministic, for reproducibility of experiments
    Parameters:
    seed: the number to set the seed to
    Return: None
    """
    # Random seed
    random.seed(seed)
    # Numpy seed
    np.random.seed(seed)
    # Torch seed
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    # os seed
    os.environ['PYTHONHASHSEED'] = str(seed)

## Dataset class

In [ ]:
class SexismDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_len=128, pad="max_length", trunc=True,rt='pt'):
        self.texts = texts.tolist()
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pad = pad
        self.trunc = trunc
        self.rt = rt

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,padding=self.pad, truncation=self.trunc,
            return_tensors=self.rt
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'id': torch.tensor(self.ids[idx], dtype=torch.long)
        }

## Metrics for subtasks 2.1 and 2.2

In [ ]:
def compute_metrics_1_2(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }



## Metrics for subtasks 2.3

In [ ]:
def compute_metrics_3(pred, lencoder):
    labels = pred.label_ids
    preds = torch.sigmoid(torch.tensor(pred.predictions)).numpy()
    preds_binary = (preds >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds_binary, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds_binary)
    # UCM must be used for the competition
    #icm= ICMWrapper(lencoder.inverse_transform(preds_binary), lencoder.inverse_transform(labels), multi=True)
    # Macro averages
    precision_macro = np.mean(precision)
    recall_macro = np.mean(recall)
    f1_macro = np.mean(f1)
    return {
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        #'ICM': icm
    }

## Discriminative fine-tuning pipeline for subtasks 2.1 and 2.2 (binary classification tasks)

In [ ]:
def sexism_classification_pipeline_task1_2(trainInfo, devInfo, task_name, mapping, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc= LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    # Prepare datasets
    #train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    #val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    train_dataset = SexismDataset(trainInfo[1], [mapping[l] for l in trainInfo[2]],[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], [mapping[l] for l in devInfo[2]], [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    # If there is a test dataset. Only for submissions to the shared-task
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame. Adapt it to the format of the competition.
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2026"]*len(predicted_labels)
        })
        submission_df.to_csv(task_name + '.csv', index=False)
        print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
        return model, submission_df
    return model, eval_results



## Discriminative fine-tuning pipeline for subtask 2.3 (multi-class multi-label classification task)

In [ ]:
def sexism_classification_pipeline_task3(trainInfo, devInfo, task_name, testInfo=None, model_name='roberta-base', nlabels=5, ptype="multi_label_classification", **args):
    # Model and Tokenizer
    labelEnc= MultiLabelBinarizer(classes=LABELS_23)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype)

    # Prepare datasets
    train_dataset = SexismDatasetMulti(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDatasetMulti(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1_macro"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        #compute_metrics=compute_metrics_3,
        compute_metrics = partial(compute_metrics_3, lencoder=labelEnc),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    if testInfo is not None:
      # Prepare test dataset for prediction
      test_dataset = SexismDatasetMulti(testInfo[1], [[0,0,0,0,0]] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

      # Predict test set labels
      predictions = trainer.predict(test_dataset)
      #predicted_labels = np.argmax(predictions.predictions, axis=1)
      predicted_probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
      predicted_labels = (predicted_probs >= 0.5).astype(int)

      # Create submission DataFrame
      submission_df = pd.DataFrame({
          'id': testInfo[0],
          'label': labelEnc.inverse_transform(predicted_labels),
          "test_case": ["EXIST2025"]*len(predicted_labels)

      })
      submission_df.to_csv(task_name + '.csv', index=False)
      print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
      return model, submission_df
    return model, eval_results

## Addressing english subtask 1

In [ ]:
# Usage Example
set_seed()
# modelname
modelname = "cardiffnlp/twitter-roberta-base-2022-154m"
# COMPLETE
#modelname = "roberta-base"
# training params
params = {
    "num_train_epochs": 10
    # "early_stopping_patience": 3
}
time0 = time.time()
m, res_en1 = sexism_classification_pipeline_task1_2(EnTrainTask1, EnDevTask1, "exist_task1_eng", MAPPING_21, None, modelname, 2, "single_label_classification", **params )
time_en1 = time.time()-time0

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

{'loss': '0.04754', 'grad_norm': '0.01062', 'learning_rate': '1.125e-05', 'epoch': '9.07'}
{'loss': '0.006917', 'grad_norm': '0.04328', 'learning_rate': '9.861e-06', 'epoch': '9.186'}
{'loss': '0.000605', 'grad_norm': '0.01839', 'learning_rate': '8.472e-06', 'epoch': '9.302'}
{'loss': '0.000506', 'grad_norm': '0.02503', 'learning_rate': '7.083e-06', 'epoch': '9.419'}
{'loss': '0.000426', 'grad_norm': '0.0118', 'learning_rate': '5.694e-06', 'epoch': '9.535'}
{'loss': '0.0005209', 'grad_norm': '0.01128', 'learning_rate': '4.306e-06', 'epoch': '9.651'}
{'loss': '0.0004738', 'grad_norm': '0.01963', 'learning_rate': '2.917e-06', 'epoch': '9.767'}
{'loss': '0.007478', 'grad_norm': '0.008722', 'learning_rate': '1.528e-06', 'epoch': '9.884'}
{'loss': '0.02971', 'grad_norm': '0.005882', 'learning_rate': '1.389e-07', 'epoch': '10'}
{'eval_loss': '1.751', 'eval_accuracy': '0.7185', 'eval_f1': '0.7767', 'eval_precision': '0.7489', 'eval_recall': '0.8068', 'eval_runtime': '2.739', 'eval_samples_per

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '744.7', 'train_samples_per_second': '18.3', 'train_steps_per_second': '1.155', 'train_loss': '0.3179', 'epoch': '10'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '1.305', 'eval_accuracy': '0.7361', 'eval_f1': '0.8', 'eval_precision': '0.7407', 'eval_recall': '0.8696', 'eval_runtime': '2.494', 'eval_samples_per_second': '136.8', 'eval_steps_per_second': '2.406', 'epoch': '10'}
Validation Results: {'eval_loss': 1.3054252862930298, 'eval_accuracy': 0.7360703812316716, 'eval_f1': 0.8, 'eval_precision': 0.7407407407407407, 'eval_recall': 0.8695652173913043, 'eval_runtime': 2.4935, 'eval_samples_per_second': 136.755, 'eval_steps_per_second': 2.406, 'epoch': 10.0}


## LoRA pipeline for subtasks 1 and 2 (binary classification tasks)

In [ ]:
######################################CHANGE###############################################
from peft import LoraConfig, get_peft_model, TaskType
###########################################################################################
def sexism_classification_pipeline_task1_2_LoRA(trainInfo, devInfo, task_name, mapping, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc = LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    ######################################CHANGE###############################################
    # Configure LoRA
    lora_config = LoraConfig(
    task_type= args.get("task_type", TaskType.SEQ_CLS),
    target_modules= args.get("target_modules", ["query", "value"]),
    r= args.get("rank", 64),  # Rank of LoRA adaptation
    lora_alpha=args.get("lora_alpha", 32),  # Scaling factor
    lora_dropout=args.get("lora_dropout", 0.1),
    bias=args.get("bias", "none")
)
    ###########################################################################################

    ######################################CHANGE###############################################
    # Prepare LoRA model
    peft_model = get_peft_model(model, lora_config)

    ###########################################################################################
    # Prepare datasets
    #train_dataset = SexismDataset(trainInfo[1], labelEnc.fit_transform(trainInfo[2]),[int(x) for x in trainInfo[0]], tokenizer )
    #val_dataset = SexismDataset(devInfo[1], labelEnc.transform(devInfo[2]), [int(x) for x in devInfo[0]], tokenizer)
    train_dataset = SexismDataset(trainInfo[1], [mapping[l] for l in trainInfo[2]],[int(x) for x in trainInfo[0]], tokenizer )
    val_dataset = SexismDataset(devInfo[1], [mapping[l] for l in devInfo[2]], [int(x) for x in devInfo[0]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results_task1_LoRA0'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        ######################################CHANGE###############################################
        # Prepare LoRA model
        model=peft_model,
        ###########################################################################################
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    ######################################CHANGE###############################################
    #Saving the new weigths for the LoRA model
    trainer.save_model('./final_best_model_LoRA')
    # Notice that, in this case only the LoRA matrices are saved.
    # The weigths for the classification head are not saved.
    ###########################################################################################

    ######################################CHANGE###############################################
    #Mixing the LoRA matrices with the weigths of the base model used
    mixModel=peft_model.merge_and_unload()
    mixModel.save_pretrained("./final_best_model_mixpeft")
    # IN this case the full model is saved.
    ###########################################################################################

    # If there is a test dataset. Only for submissions to the shared-task
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo[1], [0] * len(testInfo[1]),  [int(x) for x in testInfo[0]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo[0],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2026"]*len(predicted_labels)
        })
        submission_df.to_csv(task_name + '.csv', index=False)
        print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
        return model, submission_df
    return model, eval_results

# Experimental work

In [ ]:
# COMPLETE the experimentation

# English subtask 1 (fine-tuning and LoRA)
# English subtask 2 (fine-tuning and LoRA)
# English subtask 3 (fine-tuning and LoRA)
# Spanish subtask 1 (fine-tuning and LoRA)
# Spanish subtask 2 (fine-tuning and LoRA)
# Spanish subtask 3 (fine-tuning and LoRA)

# Total: 12 experimental results

# Some Spanish models:

#modelname = "pysentimiento/robertuito-base-uncased"
#modelname = "pysentimiento/robertuito-hate-speech"
#modelname = "pysentimiento/robertuito-base-deacc"
#modelname = "PlanTL-GOB-ES/roberta-large-bne"
#modelname = "PlanTL-GOB-ES/roberta-base-bne"
#modelname = "dccuchile/bert-base-spanish-wwm-uncased"

# Show results

In [ ]:
# COMPLETE

# Show a table comparing the performance of the different models. You can compare them across different metrics (different tables if it is needed).